In [ ]:
!pip install opencv-python numpy pywavelets torch snntorch matplotlib scipy

In [ ]:
import os
import json
import urllib.request
import zipfile
import shutil
from pathlib import Path
import cv2
import numpy as np
import pywt
import torch
import snntorch as snn
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from scipy.ndimage import zoom
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, accuracy_score
import seaborn as sns
from sklearn.model_selection import train_test_split
from torch.nn.functional import softmax

In [ ]:
# --- Data Loading Functions  ---
def download_json(url):
    """Download and load JSON file from URL."""
    try:
        print(f"Downloading JSON description from {url}...")
        response = urllib.request.urlopen(url)
        data = json.loads(response.read().decode('utf-8'))
        return data
    except Exception as e:
        print(f"Error downloading JSON: {str(e)}")
        return None

def create_sequence_prefix(seq):
    """Create standardized prefix for file naming."""
    return f"{seq['name']}_{seq['fps']}_{seq['length']}_{seq['width']}_{seq['height']}"

def process_sequence(seq, base_url_annotations="https://data.votchallenge.net/vot2015/dataset/",
                    base_url_images="https://data.votchallenge.net/sequences/"):
    """Process a single sequence, organizing files in a sequence-specific folder."""
    sequence_name = seq['name']
    prefix = create_sequence_prefix(seq)

    # Create sequence directory
    sequence_dir = os.path.join("sequences", sequence_name)
    os.makedirs(sequence_dir, exist_ok=True)

    # Create temp directory
    temp_dir = os.path.join("temp", sequence_name)
    os.makedirs(temp_dir, exist_ok=True)

    # Process groundtruth file
    try:
        # Download annotation zip
        zip_url = f"{base_url_annotations}{seq['annotations']['url']}"
        zip_path = os.path.join(temp_dir, f"{sequence_name}_annotations.zip")

        print(f"Downloading annotations for {sequence_name}...")
        urllib.request.urlretrieve(zip_url, zip_path)

        # Extract groundtruth file
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extract("groundtruth.txt", temp_dir)

        # Rename and move groundtruth file
        new_groundtruth_name = f"{prefix}_groundtruth.txt"
        os.rename(
            os.path.join(temp_dir, "groundtruth.txt"),
            os.path.join(sequence_dir, new_groundtruth_name)
        )

        print(f"Processed groundtruth for {sequence_name}")

    except Exception as e:
        print(f"Error processing groundtruth for {sequence_name}: {str(e)}")

    # Process image files
    try:
        # Download images zip
        color_info = seq['channels']['color']
        zip_url = f"{base_url_images}{color_info['uid']}.zip"
        zip_path = os.path.join(temp_dir, f"{sequence_name}_images.zip")

        print(f"Downloading images for {sequence_name}...")
        urllib.request.urlretrieve(zip_url, zip_path)

        # Extract and rename image files
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            for file_info in zip_ref.filelist:
                if file_info.filename.endswith('.jpg'):
                    # Extract file
                    zip_ref.extract(file_info, temp_dir)

                    # Rename with prefix and move to sequence directory
                    old_path = os.path.join(temp_dir, file_info.filename)
                    new_name = f"{prefix}_{file_info.filename}"
                    new_path = os.path.join(sequence_dir, new_name)

                    os.rename(old_path, new_path)

        print(f"Processed images for {sequence_name}")

    except Exception as e:
        print(f"Error processing images for {sequence_name}: {str(e)}")

    # Clean up temporary files for this sequence
    try:
        shutil.rmtree(temp_dir)
        print(f"Cleaned up temporary files for {sequence_name}")
    except Exception as e:
        print(f"Error cleaning up temporary files for {sequence_name}: {str(e)}")

def process_vot_dataset(data):
    """Process entire VOT dataset, organizing by sequence."""
    # Process each sequence
    for i, seq in enumerate(data['sequences']):
        print(f"\nProcessing sequence {i+1}/{len(data['sequences'])}: {seq['name']}")
        process_sequence(seq)

    # Final cleanup
    try:
        shutil.rmtree("temp")
        print("Cleaned up all temporary files")
    except Exception as e:
        print(f"Error during final cleanup: {str(e)}")

In [ ]:
# --- Main Class for SNN-based Movement Tracking ---
class SNNMovementTracker1:
  def __init__(self, sequence_name, image_resize=(128, 128), dwt_threshold=20,
                hidden_size=256, beta=0.1, lr=0.1):
      self.sequence_name = sequence_name
      self.image_resize = image_resize
      self.dwt_threshold = dwt_threshold
      self.hidden_size = hidden_size
      self.beta = beta
      self.lr = lr
      self.num_classes = 4  # Up, Down, Left, Right
      self.class_names = ["Up", "Down", "Left", "Right"]
      self.train_ratio = 0.7
      self.val_ratio = 0.15
      # Initialize paths
      self.sequence_dir = os.path.join("sequences", sequence_name)
      self.image_files = None
      self.ground_truth_file = None
      self.mini_video_size = 5
      # Initialize data containers
      self.frames = None
      self.ground_truth = None
      self.labels = None
      self.spike_inputs = None
      self.model = None
      self.video_size = 5 # Number of frames in each mini-video
      self.mini_videos = None
      self.train_videos = None
      self.val_videos = None
      self.test_videos = None
      self.train_labels = None
      self.val_labels = None
      self.test_labels = None
      self.train_loader = None
      self.val_loader = None
      self.test_loader = None

      # Training history
      self.train_loss_history = []
      self.val_loss_history = []
      self.train_acc_history = []
      self.val_acc_history = []

  def load_data(self):
    """Load image and groundtruth data for the sequence"""
    # Find image files
    self.image_files = sorted([f for f in os.listdir(self.sequence_dir)
                              if f.endswith(".jpg")])

    if not self.image_files:
        raise FileNotFoundError(f"No image files found in {self.sequence_dir}")

    # Find groundtruth file
    gt_files = [f for f in os.listdir(self.sequence_dir) if f.endswith("_groundtruth.txt")]
    if not gt_files:
        raise FileNotFoundError(f"No groundtruth file found in {self.sequence_dir}")
    self.ground_truth_file = os.path.join(self.sequence_dir, gt_files[0])

    # Load images
    frames = []
    frames1 =[]
    for img_file in self.image_files:
        img_path = os.path.join(self.sequence_dir, img_file)

        frame = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        frame = cv2.resize(frame, (128, 128))  # Resize for consistency
        frames.append(frame)
    frames2 = []
    spike_responses = []
    for i in range(len(frames)-1):
      frame_diff = np.abs(frames[i+1].astype(np.float32) - frames[i].astype(np.float32))

      # Apply DWT
      coeffs2 = pywt.dwt2(frame_diff, 'haar')
      LL, (LH, HL, HH) = coeffs2

      # Create richer representation by combining components
      # Scale them to enhance signal contrast
      LL_scaled = LL / LL.max() if LL.max() > 0 else LL
      LH_scaled = LH / LH.max() if LH.max() > 0 else LH
      HL_scaled = HL / HL.max() if HL.max() > 0 else HL
      combined = 0.4*LL_scaled+ 0.3*LH_scaled + 0.3*HL_scaled

      # This adjusts to the dynamic range of each frame
      adaptive_threshold = np.percentile(combined, 75)  # Top 25% of values become spikes

      spikes = np.where(combined > adaptive_threshold, 1.0, 0.0)
      spike_responses.append(spikes)
      frames2.append(combined)

    self.frames = np.array(frames2)
    self.spike_inputs = torch.tensor(spike_responses,
                                            dtype=torch.float)

    # Load ground truth
    self.ground_truth = np.loadtxt(self.ground_truth_file, delimiter=",")
    print(f'max {np.max(self.frames, axis = (0,1,2))}')
    print(f"Loaded {len(self.frames)} frames and {len(self.ground_truth)} groundtruth entries")


  def create_mini_videos(self):
    """Create mini-videos from sequential frames."""
    if self.ground_truth is None:
      raise ValueError("Ground truth data not loaded. Call load_data() first.")
    num_frames = len(self.frames)
    self.mini_videos = []
    self.labels = []

    for i in range(num_frames - self.mini_video_size):
        mini_video = self.frames[i:i + self.mini_video_size]
        self.mini_videos.append(mini_video)

        # Example labeling strategy: Based on motion direction
        prev_x = np.mean(self.ground_truth[i, [0, 2, 4, 6]])
        prev_y = np.mean(self.ground_truth[i, [1, 3, 5, 7]])
        curr_x = np.mean(self.ground_truth[i + self.mini_video_size - 1, [0, 2, 4, 6]])
        curr_y = np.mean(self.ground_truth[i + self.mini_video_size - 1, [1, 3, 5, 7]])

        dx = curr_x - prev_x
        dy = curr_y - prev_y

        if abs(dy) > abs(dx):  # Vertical motion
            label = 0 if dy < 0 else 1  # Up or Down
        else:  # Horizontal motion
            label = 2 if dx < 0 else 3  # Left or Right

        self.labels.append(label)

    self.mini_videos = np.array(self.mini_videos, dtype=np.float32)
    self.labels = np.array(self.labels, dtype=np.int64)
    print(f"Created {len(self.mini_videos)} mini-videos of {self.mini_video_size} frames each")

    # Print distribution of labels
    unique_labels, counts = np.unique(self.labels, return_counts=True)
    print("Label distribution:")
    for i, count in zip(unique_labels, counts):
        print(f"  {self.class_names[i]}: {count} ({count/len(self.labels)*100:.1f}%)")


  def split_dataset(self):
      """Split mini-videos into training, validation, and test sets."""
      train_videos, temp_videos, train_labels, temp_labels = train_test_split(
          self.mini_videos, self.labels, test_size=1 - self.train_ratio, random_state=42, stratify=self.labels)

      val_size = self.val_ratio / (1 - self.train_ratio)
      val_videos, test_videos, val_labels, test_labels = train_test_split(
          temp_videos, temp_labels, test_size=1 - val_size, random_state=42, stratify=temp_labels)

      # Convert to tensors
      self.train_videos = torch.tensor(train_videos, dtype=torch.float32)
      self.val_videos = torch.tensor(val_videos, dtype=torch.float32)
      self.test_videos = torch.tensor(test_videos, dtype=torch.float32)

      self.train_labels = torch.tensor(train_labels, dtype=torch.long)
      self.val_labels = torch.tensor(val_labels, dtype=torch.long)
      self.test_labels = torch.tensor(test_labels, dtype=torch.long)

      print(f"Dataset split: {len(self.train_videos)} train, {len(self.val_videos)} val, {len(self.test_videos)} test")

  def get_data_loaders(self, batch_size=16):
      """Create PyTorch DataLoaders."""

      # Print dataset sizes to help debug any issues
      print(f"Dataset sizes - Train: {len(self.train_videos)}, Val: {len(self.val_videos)}, Test: {len(self.test_videos)}")

      # Check if batch size is appropriate for dataset sizes
      if batch_size > len(self.val_videos):
        print(f"Warning: Batch size ({batch_size}) is larger than validation set size ({len(self.val_videos)}).")
        print(f"Consider reducing batch size or using different train/val split ratios.")

      train_dataset = torch.utils.data.TensorDataset(self.train_videos, self.train_labels)
      val_dataset = torch.utils.data.TensorDataset(self.val_videos, self.val_labels)
      test_dataset = torch.utils.data.TensorDataset(self.test_videos, self.test_labels)

      train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
      val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, drop_last=False)
      test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, drop_last=False)

      return train_loader, val_loader, test_loader

  def compute_class_weights(self,lst):
    labels = lst[0]
    num_classes = lst[1]
    """Compute inverse frequency class weights"""
    class_counts = torch.bincount(labels, minlength=num_classes)
    total_samples = class_counts.sum()
    class_weights = total_samples / (class_counts + 1e-6)  # Avoid division by zero
    class_weights /= class_weights.sum()  # Normalize
    return class_weights

  def create_model(self, beta = 0.6, lr = 0.2 ):
    """Define the SNN model architecture"""

    # Account for the additional features (density and variance)
    #input_size = (self.image_resize[0] * self.image_resize[1]) + 2
    #print(input_size)
    class SNNModel(nn.Module):
        def __init__(self, input_size =(128*128)//4, hidden_size = 200, output_size = 4, beta = beta):
            super().__init__()
            # Input normalization
            #self.bn_input = nn.BatchNorm1d(4096)

            # Two hidden layers architecture
            #print(input_size)
            self.fc1 = nn.Linear(input_size, hidden_size)
            self.lif1 = snn.Leaky(beta=beta)
            self.bn1 = nn.BatchNorm1d(hidden_size)
            self.dropout1 = nn.Dropout(0.3)

            self.fc2 = nn.Linear(hidden_size, hidden_size // 2)
            self.lif2 = snn.Leaky(beta=beta)
            self.bn2 = nn.BatchNorm1d(hidden_size // 2)
            self.dropout2 = nn.Dropout(0.3)

            # Output layer
            self.fc3 = nn.Linear(hidden_size // 2, output_size)
            self.lif3 = snn.Leaky(beta=beta)

        def forward(self, x):
            # Initialize membrane potentials
            mem1 = self.lif1.init_leaky()
            mem2 = self.lif2.init_leaky()
            mem3 = self.lif3.init_leaky()

            batch_size = x.shape[0]

            for t in range(5):

              frame_batch = x[:, t, :, :]  # Shape: [batch_size, frame_no, H, W]

              flattened_tensor = frame_batch.view(batch_size, -1)

              # First hidden layer
              cur1 = self.fc1(flattened_tensor)
              spk1, mem1 = self.lif1(cur1, mem1)
              spk1 = self.bn1(spk1)
              spk1 = self.dropout1(spk1)

              # Second hidden layer
              cur2 = self.fc2(spk1)
              spk2, mem2 = self.lif2(cur2, mem2)
              spk2 = self.bn2(spk2)
              spk2 = self.dropout2(spk2)

              # Output layer
              cur3 = self.fc3(spk2)
              spk3, mem3 = self.lif3(cur3, mem3)
            return spk3, [mem3, spk3]


    weights = self.compute_class_weights([self.train_labels, 4])
    self.model = SNNModel()

    self.optimizer = optim.SGD(self.model.parameters(), lr=lr, momentum=0.9)
    self.loss_fn = nn.CrossEntropyLoss(label_smoothing=0.2, weight=weights)
    print(f"Created SNN model")

  def train(self, num_epochs=10, batch_size=16):
    """Train the SNN model with adaptive learning rate and model saving"""
    if self.model is None:
      raise ValueError("Model not created. Call create_model() first.")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on {device}")

    self.model.to(device)

    # Create data loaders
    self.train_loader, self.val_loader, self.test_loader = self.get_data_loaders(batch_size)
    train_loader = self.train_loader
    val_loader = self.val_loader

    # Initialize history trackers
    self.train_loss_history = []
    self.val_loss_history = []
    self.train_acc_history = []
    self.val_acc_history = []

    # Setup adaptive learning rate scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
      self.optimizer,
      mode='min',
      factor=0.75,
      patience=15,
      verbose=True,
      min_lr=1e-6
    )

    # Track best model and performance
    best_val_loss = float('inf')
    best_epoch = -1
    patience_counter = 0
    patience = 100  # Early stopping patience - will stop after 100 epochs without improvement

    for epoch in range(num_epochs):
      # Training
      self.model.train()
      train_loss = 0.0
      train_correct = 0
      train_total = 0

      for inputs, targets in train_loader:
        inputs, targets = inputs.to(device), targets.to(device)

        self.optimizer.zero_grad()

        # Forward pass
        spk_out, lst = self.model(inputs)

        logits = spk_out
        loss = self.loss_fn(logits, targets)

        # Backward pass
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)  # Prevent exploding gradients

        self.optimizer.step()

        # Calculate training accuracy
        _, preds = torch.max(spk_out, 1)
        train_correct += (preds == targets).sum().item()
        train_total += targets.size(0)
        train_loss += loss.item() * inputs.size(0)

      # Prevent division by zero
      if train_total > 0:
        train_loss /= train_total
        train_accuracy = 100.0 * train_correct / train_total
      else:
        train_loss = float('inf')
        train_accuracy = 0.0
        print("Warning: No training samples were processed. Batch size may be too large for training set.")
      self.train_loss_history.append(train_loss)
      self.train_acc_history.append(train_accuracy)

      # Validation
      self.model.eval()
      val_loss = 0.0
      val_correct = 0
      val_total = 0

      with torch.no_grad():
        for inputs, targets in val_loader:
          inputs, targets = inputs.to(device), targets.to(device)

          # Forward pass
          spk_out, _ = self.model(inputs)
          loss = self.loss_fn(spk_out, targets)

          # Calculate validation accuracy
          _, preds = torch.max(spk_out, 1)
          val_correct += (preds == targets).sum().item()
          val_total += targets.size(0)
          val_loss += loss.item() * inputs.size(0)

      # Prevent division by zero
      if val_total > 0:
        val_loss /= val_total
        val_accuracy = 100.0 * val_correct / val_total
      else:
        val_loss = float('inf')  # Set to infinity if no validation samples processed
        val_accuracy = 0.0
        print("Warning: No validation samples were processed. Batch size may be too large for validation set.")
      self.val_loss_history.append(val_loss)
      self.val_acc_history.append(val_accuracy)

      # Update learning rate based on validation loss
      scheduler.step(val_loss)

      # Print epoch results (one line, every 5 epochs)
      current_lr = self.optimizer.param_groups[0]['lr']
      if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.2f}%, Val Loss: {val_loss:.4f}, Val Acc: {val_accuracy:.2f}%, LR: {current_lr:.6f}")

      # Early stopping check and best model saving
      if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        patience_counter = 0

        # Create model save directory if it doesn't exist
        os.makedirs("models", exist_ok=True)

        # Save model with descriptive filename including sequence name
        model_path = f"models/{self.sequence_name}_snn_model_epoch{epoch+1}_valLoss{val_loss:.4f}.pth"
        torch.save({
          'epoch': epoch,
          'model_state_dict': self.model.state_dict(),
          'optimizer_state_dict': self.optimizer.state_dict(),
          'scheduler_state_dict': scheduler.state_dict(),
          'train_loss': train_loss,
          'val_loss': val_loss,
          'train_acc': train_accuracy,
          'val_acc': val_accuracy,
          'best_val_loss': best_val_loss,
          'sequence_name': self.sequence_name
        }, model_path)

        # Save a permanent "best" version for easier reference
        best_model_path = f"models/{self.sequence_name}_snn_model_best.pth"
        torch.save({
          'epoch': epoch,
          'model_state_dict': self.model.state_dict(),
          'optimizer_state_dict': self.optimizer.state_dict(),
          'scheduler_state_dict': scheduler.state_dict(),
          'train_loss': train_loss,
          'val_loss': val_loss,
          'train_acc': train_accuracy,
          'val_acc': val_accuracy,
          'best_val_loss': best_val_loss,
          'sequence_name': self.sequence_name
        }, best_model_path)
      else:
        patience_counter += 1
        if patience_counter >= patience:
          print(f"Early stopping at epoch {epoch+1}. No improvement for {patience} epochs.")
          break

    print(f"\nTraining complete. Best model saved at epoch {best_epoch} with validation loss: {best_val_loss:.4f}")

    # Load the best model for evaluation
    best_model_path = f"models/{self.sequence_name}_snn_model_best.pth"
    checkpoint = torch.load(best_model_path)
    self.model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded best model from epoch {checkpoint['epoch']+1} for evaluation.")

  def evaluate(self):
    """Evaluate the model on test data and produce visualizations

    Returns:
        dict: Dictionary containing all evaluation metrics
    """
    if self.model is None:
        raise ValueError("Model not trained. Call train() first.")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    self.model.to(device)
    self.model.eval()
    test_loader = self.test_loader


    # Compute accuracy and gather predictions
    correct = 0
    total = 0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)

            # Forward pass
            spk_out, _ = self.model(inputs)
            #print(f' test out: {spk_out}')
            # Get predictions
            _, preds = torch.max(spk_out, 1)

            # Update statistics
            total += targets.size(0)
            correct += (preds == targets).sum().item()

            # Store for confusion matrix
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())

     # Calculate accuracy, precision, recall, and F1-score
        accuracy = 100.0 * correct / total
        precision, recall, f1, _ = precision_recall_fscore_support(
            all_targets, all_preds, average=None, labels=range(self.num_classes)
        )

        # Overall metrics (macro average)
        overall_precision = np.mean(precision)
        overall_recall = np.mean(recall)
        overall_f1 = np.mean(f1)

        # Calculate confusion matrix
        cm = confusion_matrix(all_targets, all_preds)

        # Print results
        print(f"Test Accuracy: {accuracy:.2f}%")
        print(f"Overall Precision: {overall_precision*100:.2f}%")
        print(f"Overall Recall: {overall_recall*100:.2f}%")
        print(f"Overall F1 Score: {overall_f1*100:.2f}%")

        print("\nPer-class metrics:")
        for i in range(self.num_classes):
            print(f"  {self.class_names[i]}:")
            print(f"    Precision: {precision[i]*100:.2f}%")
            print(f"    Recall: {recall[i]*100:.2f}%")
            print(f"    F1 Score: {f1[i]*100:.2f}%")

        # Plot results
        plt.figure(figsize=(20, 12))

        # Add sequence name to the main figure title
        plt.suptitle(f"Movement Tracking Results for Sequence: {self.sequence_name}", fontsize=16, y=0.98)

        # Plot training and validation loss & accuracy
        plt.subplot(2, 2, 1)
        plt.plot(self.train_loss_history, 'b-', label='Training Loss')
        plt.plot(self.val_loss_history, 'r-', label='Validation Loss')
        plt.title('Learning Curves')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True, alpha=0.3)

        plt.subplot(2, 2, 2)
        plt.plot(self.train_acc_history, 'b-', label='Training Accuracy')
        plt.plot(self.val_acc_history, 'r-', label='Validation Accuracy')
        plt.title('Accuracy Curves')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy (%)')
        plt.legend()
        plt.grid(True, alpha=0.3)

        # Plot confusion matrix
        plt.subplot(2, 2, 3)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=self.class_names,
                    yticklabels=self.class_names)
        plt.title(f'Confusion Matrix\nAcc: {accuracy:.1f}%, Prec: {overall_precision*100:.1f}%\nRec: {overall_recall*100:.1f}%, F1: {overall_f1*100:.1f}%')
        plt.xlabel('Predicted Label')
        plt.ylabel('True Label')

        # Plot precision, recall, and F1-score
        plt.subplot(2, 2, 4)
        x = np.arange(len(self.class_names))
        width = 0.25

        # Calculate per-class accuracy from confusion matrix
        per_class_accuracy = np.zeros(self.num_classes)
        for i in range(self.num_classes):
            per_class_accuracy[i] = 100.0 * cm[i, i] / np.sum(cm[i, :]) if np.sum(cm[i, :]) > 0 else 0

        # Create grouped bar chart with 4 metrics
        bar_width = 0.2  # Narrower bars to fit 4 groups
        plt.bar(x - 1.5*bar_width, per_class_accuracy, bar_width, label='Accuracy')
        plt.bar(x - 0.5*bar_width, precision * 100, bar_width, label='Precision')
        plt.bar(x + 0.5*bar_width, recall * 100, bar_width, label='Recall')
        plt.bar(x + 1.5*bar_width, f1 * 100, bar_width, label='F1 Score')

        plt.title('Classification Metrics by Class')
        plt.xlabel('Class')
        plt.ylabel('Score (%)')
        plt.xticks(x, self.class_names)
        plt.ylim(0, 100)
        plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=4)
        plt.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig(f"{self.sequence_name}_sandra_results.png", dpi=300, bbox_inches='tight')
        plt.show()

        # Calculate metrics dictionary for return
        metrics = {
            'accuracy': accuracy,
            'precision': precision * 100,
            'recall': recall * 100,
            'f1': f1 * 100,
            'confusion_matrix': cm
        }

    return metrics

# --- Main Execution Logic ---
def main():
  # Import necessary modules for this function
  import pandas as pd
  from datetime import datetime

  # Step 1: Setup directories
  os.makedirs("sequences", exist_ok=True)
  os.makedirs("temp", exist_ok=True)
  os.makedirs("models", exist_ok=True)
  os.makedirs("results", exist_ok=True)

  # Step 2: Get all sequence names
  sequence_names = fetch_vot_sequence_names()

  if not sequence_names:
    print("Failed to retrieve sequence names. Exiting.")
    return

  print(f"Found {len(sequence_names)} sequences to process")

  # Create DataFrame to store results
  results_df = pd.DataFrame(columns=[
    'sequence_name', 'total_frames', 'count_up', 'count_down', 'count_left', 'count_right',
    'accuracy', 'precision_up', 'precision_down', 'precision_left', 'precision_right',
    'recall_up', 'recall_down', 'recall_left', 'recall_right',
    'f1_up', 'f1_down', 'f1_left', 'f1_right',
    'overall_precision', 'overall_recall', 'overall_f1'
  ])

  # Get current timestamp for unique results file
  timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

  # Process each sequence
  for i, sequence_name in enumerate(sequence_names):
    try:
      print(f"\n[{i+1}/{len(sequence_names)}] Processing sequence: {sequence_name}")

      # Download the dataset JSON
      json_url = "https://data.votchallenge.net/vot2015/dataset/description.json"
      data = download_json(json_url)

      if not data:
        print(f"Failed to download dataset description for {sequence_name}. Skipping.")
        continue

      # Find the sequence in the data
      seq = next((s for s in data['sequences'] if s['name'] == sequence_name), None)

      if not seq:
        print(f"Sequence {sequence_name} not found in dataset. Skipping.")
        continue

      # Process the sequence
      try:
        process_sequence(seq)
      except Exception as e:
        print(f"Error processing sequence {sequence_name}: {e}. Skipping.")
        continue

      # Create and train the movement tracker
      tracker = SNNMovementTracker1(sequence_name,
                                    image_resize=(128, 128),
                                    dwt_threshold=20,
                                    hidden_size=256)

      try:
        # Process the data
        tracker.load_data()
        tracker.create_mini_videos()
        tracker.split_dataset()

        # Train and evaluate the model
        tracker.create_model(beta=0.69, lr=0.07)
        tracker.train(num_epochs=500, batch_size=16)
        metrics = tracker.evaluate()

        # Calculate direction counts from the labels
        unique_labels, label_counts = np.unique(tracker.labels, return_counts=True)

        # Initialize counts for each direction
        counts_by_direction = {0: 0, 1: 0, 2: 0, 3: 0}  # Up, Down, Left, Right

        # Fill in the actual counts
        for label, count in zip(unique_labels, label_counts):
            counts_by_direction[label] = count

        # Extract metrics for DataFrame
        sequence_results = {
          'sequence_name': sequence_name,
          'total_frames': len(tracker.frames) + 1,  # Adding 1 because frames is derived from frame differences
          'count_up': counts_by_direction[0],
          'count_down': counts_by_direction[1],
          'count_left': counts_by_direction[2],
          'count_right': counts_by_direction[3],
          'accuracy': metrics['accuracy'],
          'precision_up': metrics['precision'][0],
          'precision_down': metrics['precision'][1],
          'precision_left': metrics['precision'][2],
          'precision_right': metrics['precision'][3],
          'recall_up': metrics['recall'][0],
          'recall_down': metrics['recall'][1],
          'recall_left': metrics['recall'][2],
          'recall_right': metrics['recall'][3],
          'f1_up': metrics['f1'][0],
          'f1_down': metrics['f1'][1],
          'f1_left': metrics['f1'][2],
          'f1_right': metrics['f1'][3],
          'overall_precision': np.mean(metrics['precision']),
          'overall_recall': np.mean(metrics['recall']),
          'overall_f1': np.mean(metrics['f1'])
        }

        # Add to results DataFrame
        results_df = pd.concat([results_df, pd.DataFrame([sequence_results])], ignore_index=True)

        # Save intermediate results after each sequence
        results_df.to_csv(f"results/snn_tracking_results_{timestamp}.csv", index=False)

        print(f"Sequence {sequence_name} completed with accuracy: {metrics['accuracy']:.2f}%")

      except Exception as e:
        print(f"Error training/evaluating sequence {sequence_name}: {e}. Skipping.")
        continue

    except Exception as e:
      print(f"Unexpected error with sequence {sequence_name}: {e}. Continuing with next sequence.")
      continue

  # After all sequences are processed, save and display final results
  results_df.to_csv(f"results/snn_tracking_results_final_{timestamp}.csv", index=False)

  print("\n===== FINAL RESULTS =====")
  print(results_df.describe())

  # Generate summary plot
  try:
    plt.figure(figsize=(15, 10))

    # Sort by accuracy
    sorted_df = results_df.sort_values('accuracy', ascending=False)

    # Plot accuracies
    plt.subplot(2, 1, 1)
    plt.bar(sorted_df['sequence_name'], sorted_df['accuracy'], color='skyblue')
    plt.xlabel('Sequence')
    plt.ylabel('Accuracy (%)')
    plt.title('Accuracy by Sequence')
    plt.xticks(rotation=90)
    plt.grid(axis='y', alpha=0.3)

    # Plot F1 scores
    plt.subplot(2, 1, 2)
    plt.bar(sorted_df['sequence_name'], sorted_df['overall_f1'], color='lightgreen')
    plt.xlabel('Sequence')
    plt.ylabel('F1 Score (%)')
    plt.title('F1 Score by Sequence')
    plt.xticks(rotation=90)
    plt.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"results/overall_performance_summary_{timestamp}.png", dpi=300, bbox_inches='tight')
    plt.show()

  except Exception as e:
    print(f"Error creating summary plot: {e}")

  print(f"All results saved to results/snn_tracking_results_final_{timestamp}.csv")

import json
import requests

def fetch_vot_sequence_names(url="https://data.votchallenge.net/vot2015/dataset/description.json"):
    """
    Fetch sequence names from the VOT2015 Challenge JSON file URL.

    Args:
        url (str): URL to the VOT2015 dataset description JSON

    Returns:
        list: List of all sequence names
    """
    try:
        # Fetch the JSON data from the URL
        response = requests.get(url)
        response.raise_for_status()  # Raise an exception for HTTP errors

        # Parse the JSON data
        data = response.json()

        # Extract sequence names from the sequences array
        sequence_names = [sequence['name'] for sequence in data['sequences']]

        # Print in the format requested
        print(f"sequence_names={sequence_names}")
        #sequence_names = ['bmx', 'handball1', 'leaves', 'marching', 'singer1']
        return sequence_names

    except requests.exceptions.RequestException as e:
        print(f"Error fetching data: {e}")
        return []
    except json.JSONDecodeError as e:
        print(f"Error parsing JSON data: {e}")
        return []
    except KeyError as e:
        print(f"Error accessing JSON structure: {e}")
        return []

if __name__ == "__main__":
    main()

In [ ]:
import os
import glob
import zipfile
from google.colab import files

# Define the directory to search for PNG files
directory = '/content'

# Create a list of all PNG files in the directory
png_files = glob.glob(os.path.join(directory, '*.png'))

# Check if any PNG files were found
if png_files:
    # Create a zip file
    zip_filename = 'png_files.zip'

    with zipfile.ZipFile(zip_filename, 'w') as zipf:
        # Add each PNG file to the zip
        for png_file in png_files:
            # Add file to zip (arcname extracts just the filename, not the full path)
            zipf.write(png_file, arcname=os.path.basename(png_file))

    # Download the zip file
    files.download(zip_filename)
    print(f"Created and downloaded {zip_filename} with {len(png_files)} PNG files")
else:
    print("No PNG files found in /content directory")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

In [ ]:
def compute_nsd(df):
    # Extract relevant columns
    total_frames = df['total_frames']
    count_up = df['count_up']
    count_down = df['count_down']
    count_left = df['count_left']
    count_right = df['count_right']

    # Calculate ideal count per direction (μ)
    mu = total_frames / 4

    # Calculate squared differences from ideal
    squared_diff_up = (count_up - mu) ** 2
    squared_diff_down = (count_down - mu) ** 2
    squared_diff_left = (count_left - mu) ** 2
    squared_diff_right = (count_right - mu) ** 2

    # Calculate average squared difference
    avg_squared_diff = (squared_diff_up + squared_diff_down + squared_diff_left + squared_diff_right) / 4

    # Calculate standard deviation
    std_dev = np.sqrt(avg_squared_diff)

    # Calculate NSD
    nsd = std_dev / mu

    # Add NSD to dataframe
    df['nsd'] = nsd

    return df

def compute_entropy(df):
    """
    Calculate entropy for each row in the dataframe based on direction counts
    """
    # Create a new dataframe to store results
    result_df = df.copy()

    # Function to safely calculate p*log2(p)
    def safe_entropy_term(p):
        return np.where(p > 0, -p * np.log2(p), 0)

    # Calculate probabilities (element-wise)
    p_up = df['count_up'] / df['total_frames']
    p_down = df['count_down'] / df['total_frames']
    p_left = df['count_left'] / df['total_frames']
    p_right = df['count_right'] / df['total_frames']

    # Calculate entropy terms and sum them
    entropy = safe_entropy_term(p_up) + safe_entropy_term(p_down) + \
              safe_entropy_term(p_left) + safe_entropy_term(p_right)

    # Add entropy to dataframe
    result_df['entropy'] = entropy

    return result_df


##########################################3

# Path to your tracking results CSV
#csv_path = '/content/results/snn_tracking_results_final_20250315_123723.csv'
#csv_path = '/content/snn_tracking_results_final_20250315_123723.csv'
csv_path = '/content/results/snn_tracking_results_20250316_153119.csv'

# Load the CSV file
df = pd.read_csv(csv_path)

# Print the column names
print(df.columns)

# Compute NSD and add it to the dataframe
df = compute_nsd(df)

# Compute entropy and add it to the dataframe
df = compute_entropy(df)

In [ ]:
df.describe()

In [ ]:
%matplotlib inline

metrics_to_plot = ['accuracy', 'overall_precision', 'overall_recall', 'overall_f1']

# Check data formatting
print(f"Number of rows in DataFrame: {len(df)}")
print(f"Data types: {df[metrics_to_plot].dtypes}")
print(f"Data range: Min={df[metrics_to_plot].min().min()}, Max={df[metrics_to_plot].max().max()}")

# Create a melted version of the data for seaborn boxplot
melted_data = pd.melt(df[metrics_to_plot], var_name='Metric', value_name='Value')

# Set the figure size and style
plt.figure(figsize=(12, 8))
sns.set_style("whitegrid")
sns.set_context("notebook", font_scale=1.2)

# Create the boxplot - force the showfliers parameter to True to show outliers
ax = sns.boxplot(x='Metric', y='Value', data=melted_data,
                palette='viridis',
                width=0.5,
                showfliers=True,
                medianprops={"color": "red", "linewidth": 2},
                flierprops={"marker": "o", "markerfacecolor": "red", "markersize": 8})

# Add individual data points for better visibility
# Reduce the size and opacity so they don't overwhelm the plot
sns.stripplot(x='Metric', y='Value', data=melted_data,
              size=4, color='black', alpha=0.3, jitter=True)

# Customize the plot
plt.title('Overall Performance Metrics', fontsize=18, pad=20)
plt.xlabel('Metric', fontsize=16, labelpad=15)
plt.ylabel('Value (%)', fontsize=16, labelpad=15)

# Based on your summary statistics, your data ranges from 0 to approximately
# 92%, so set the y-axis limits to accommodate this with some padding
plt.ylim(-5, 105)

# Add horizontal lines for better reference
plt.axhline(y=50, color='gray', linestyle='--', alpha=0.5)
plt.axhline(y=75, color='gray', linestyle='--', alpha=0.5)
plt.axhline(y=25, color='gray', linestyle='--', alpha=0.5)

# Rename the x-axis labels to be more readable
ax.set_xticklabels(['Accuracy', 'Precision', 'Recall', 'F1 Score'], fontsize=14)

# Annotate with summary statistics on the plot
for i, metric in enumerate(metrics_to_plot):
    median = df[metric].median()
    plt.text(i, -3, f"Median: {median:.1f}", ha='center', fontsize=11)

# Add grid for better readability, but only on the y-axis
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Ensure the plot is displayed with good spacing
plt.tight_layout()
plt.show()

# Let's also create a violin plot which might better show the distribution
plt.figure(figsize=(12, 8))
sns.set_style("whitegrid")

# Create violin plot with box plot inside
ax = sns.violinplot(x='Metric', y='Value', data=melted_data,
                   palette='viridis', inner='box', cut=0)

plt.title('Distribution of Performance Metrics', fontsize=18)
plt.xlabel('Metric', fontsize=16)
plt.ylabel('Value (%)', fontsize=16)
plt.ylim(-5, 105)
ax.set_xticklabels(['Accuracy', 'Precision', 'Recall', 'F1 Score'], fontsize=14)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
%matplotlib inline

# Create a figure with 4 subplots (2x2 grid)
fig, axes = plt.subplots(2, 2, figsize=(20, 16))
plt.subplots_adjust(hspace=0.3, wspace=0.3)

# Define the directions and their corresponding metrics
directions = ['up', 'down', 'left', 'right']
metric_types = ['precision', 'recall', 'f1']

# Define colors for consistency
colors = ['#4287f5', '#42c9f5', '#42f5a7', '#f5e642']

# Loop through each direction and create subplot
for idx, direction in enumerate(directions):
    # Calculate subplot position (row, column)
    row = idx // 2
    col = idx % 2

    # Get current axis
    ax = axes[row, col]

    # Collect metrics for this direction
    direction_metrics = [f'{metric_type}_{direction}' for metric_type in metric_types]

    # Create melted data for this direction
    melted_data = pd.melt(df[direction_metrics], var_name='Metric', value_name='Value')

    # Create boxplot
    sns.boxplot(x='Metric', y='Value', data=melted_data,
                palette=colors[:len(metric_types)],
                width=0.5,
                showfliers=True,
                medianprops={"color": "red", "linewidth": 2},
                flierprops={"marker": "o", "markerfacecolor": "red", "markersize": 6},
                ax=ax)

    # Add individual data points
    sns.stripplot(x='Metric', y='Value', data=melted_data,
                  size=3, color='black', alpha=0.3, jitter=True, ax=ax)

    # Set labels and title for this subplot
    ax.set_title(f'{direction.capitalize()} Direction Metrics', fontsize=16, pad=15)
    ax.set_xlabel('Metric Type', fontsize=14, labelpad=10)
    ax.set_ylabel('Value (%)', fontsize=14, labelpad=10)

    # Set y-axis limits
    ax.set_ylim(-5, 105)

    # Add horizontal reference lines
    ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5)
    ax.axhline(y=75, color='gray', linestyle='--', alpha=0.5)
    ax.axhline(y=25, color='gray', linestyle='--', alpha=0.5)

    # Rename x-tick labels to be more readable
    ax.set_xticklabels(['Precision', 'Recall', 'F1 Score'], fontsize=12)

    # Add grid
    ax.grid(axis='y', linestyle='--', alpha=0.7)

    # Annotate with median values
    for i, metric in enumerate(direction_metrics):
        median = df[metric].median()
        ax.text(i, -3, f"Median: {median:.1f}", ha='center', fontsize=10)

# Add an overall title to the figure
fig.suptitle('Direction-Specific Performance Metrics', fontsize=20, y=0.98)

# Ensure the plot is displayed with good spacing
plt.tight_layout()
plt.subplots_adjust(top=0.93)  # Make room for the suptitle
plt.show()

# Set up a figure with 3 subplots (one for each metric type)
fig, axes = plt.subplots(1, 3, figsize=(20, 8))
plt.subplots_adjust(wspace=0.3)

# For each metric type (precision, recall, f1)
for i, metric_type in enumerate(metric_types):
    # Get current axis
    ax = axes[i]

    # Collect direction-specific metrics of this type
    type_metrics = [f'{metric_type}_{direction}' for direction in directions]

    # Create melted data for this metric type
    melted_data = pd.melt(df[type_metrics], var_name='Direction', value_name='Value')

    # Create boxplot
    sns.boxplot(x='Direction', y='Value', data=melted_data,
                palette=colors,
                width=0.5,
                showfliers=True,
                medianprops={"color": "red", "linewidth": 2},
                flierprops={"marker": "o", "markerfacecolor": "red", "markersize": 6},
                ax=ax)

    # Add individual data points
    sns.stripplot(x='Direction', y='Value', data=melted_data,
                  size=3, color='black', alpha=0.3, jitter=True, ax=ax)

    # Set labels and title
    ax.set_title(f'{metric_type.capitalize()} by Direction', fontsize=16, pad=15)
    ax.set_xlabel('Direction', fontsize=14, labelpad=10)
    ax.set_ylabel('Value (%)', fontsize=14, labelpad=10)

    # Set y-axis limits
    ax.set_ylim(-5, 105)

    # Add horizontal reference lines
    ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5)
    ax.axhline(y=75, color='gray', linestyle='--', alpha=0.5)
    ax.axhline(y=25, color='gray', linestyle='--', alpha=0.5)

    # Update x-tick labels
    ax.set_xticklabels(['Up', 'Down', 'Left', 'Right'], fontsize=12)

    # Add grid
    ax.grid(axis='y', linestyle='--', alpha=0.7)

    # Annotate with median values
    for j, metric in enumerate(type_metrics):
        median = df[metric].median()
        ax.text(j, -3, f"Median: {median:.1f}", ha='center', fontsize=10)

# Add an overall title
fig.suptitle('Comparing Directions for Each Metric Type', fontsize=20, y=0.98)

plt.tight_layout()
plt.subplots_adjust(top=0.9)
plt.show()

In [ ]:
%matplotlib inline

# Create a figure with 4 subplots (2x2 grid)
fig, axes = plt.subplots(2, 2, figsize=(20, 16))
plt.subplots_adjust(hspace=0.3, wspace=0.3)

# Define the directions and their corresponding metrics
directions = ['up', 'down', 'left', 'right']
metric_types = ['precision', 'recall', 'f1']

# Define colors for consistency
colors = ['#4287f5', '#42c9f5', '#42f5a7']

# Flatten the axes array for easier indexing
axes = axes.flatten()

# Loop through each direction and create subplot
for idx, direction in enumerate(directions):
    # Get current axis
    ax = axes[idx]

    # Collect metrics for this direction
    direction_metrics = [f'{metric_type}_{direction}' for metric_type in metric_types]

    # Create melted data for this direction
    melted_data = pd.melt(df[direction_metrics], var_name='Metric', value_name='Value')

    # Create boxplot
    sns.boxplot(x='Metric', y='Value', data=melted_data,
                palette=colors,
                width=0.5,
                showfliers=True,
                medianprops={"color": "red", "linewidth": 2},
                flierprops={"marker": "o", "markerfacecolor": "red", "markersize": 6},
                ax=ax)

    # Add individual data points
    sns.stripplot(x='Metric', y='Value', data=melted_data,
                  size=3, color='black', alpha=0.3, jitter=True, ax=ax)

    # Set labels and title for this subplot
    ax.set_title(f'{direction.capitalize()} Direction Metrics', fontsize=16, pad=15)
    ax.set_xlabel('Metric Type', fontsize=14, labelpad=10)
    ax.set_ylabel('Value (%)', fontsize=14, labelpad=10)

    # Set y-axis limits
    ax.set_ylim(-5, 105)

    # Add horizontal reference lines
    ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5)
    ax.axhline(y=75, color='gray', linestyle='--', alpha=0.5)
    ax.axhline(y=25, color='gray', linestyle='--', alpha=0.5)

    # Rename x-tick labels to be more readable
    ax.set_xticklabels(['Precision', 'Recall', 'F1 Score'], fontsize=12)

    # Add grid
    ax.grid(axis='y', linestyle='--', alpha=0.7)

    # Annotate with median values
    for i, metric in enumerate(direction_metrics):
        median = df[metric].median()
        ax.text(i, -3, f"Median: {median:.1f}", ha='center', fontsize=10)

# Add an overall title to the figure
fig.suptitle('Direction-Specific Performance Metrics', fontsize=20, y=0.98)

# Ensure the plot is displayed with good spacing
plt.tight_layout()
plt.subplots_adjust(top=0.93)  # Make room for the suptitle
plt.show()

# Print summary statistics for all direction metrics
print("\nSummary Statistics for Direction Metrics:")
direction_metrics_all = []
for direction in directions:
    for metric_type in metric_types:
        direction_metrics_all.append(f'{metric_type}_{direction}')

print(df[direction_metrics_all].describe().round(1))

In [ ]:
# Creating the scatter plot
plt.figure(figsize=(10, 6))
sns.scatterplot(x='entropy', y='overall_f1', data=df, s=100, alpha=0.7)

# Adding a trend line
sns.regplot(x='entropy', y='overall_f1', data=df, scatter=False, color='green')

# Adding labels and title
plt.xlabel('Entropy', fontsize=12)
plt.ylabel('Overall F1 Score', fontsize=12)
plt.title('Relationship Between Dataset Balance (Entropy) and Model Performance (F1)', fontsize=14)

# Add reference line for maximum entropy (2.0)
plt.axvline(x=2.0, color='red', linestyle='--', alpha=0.5)
plt.text(2.01, plt.ylim()[0] + 0.02, 'Max Entropy (2.0)', rotation=90, va='bottom')

# Improve the appearance
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()

# Save the figure
plt.savefig('f1_vs_entropy.png', dpi=300)

# Show the plot
plt.show()

In [ ]:
# Create the figure and axis
plt.figure(figsize=(12, 8))

# Create a scatter plot
scatter = plt.scatter(df['total_frames'], df['nsd'],
                      c=df['overall_f1'],
                      cmap='cividis',  # Using cividis colormap (colorblind friendly)
                      s=100,           # Size of markers
                      alpha=0.8,       # Transparency
                      edgecolors='black')

# Add a color bar
cbar = plt.colorbar(scatter)
cbar.set_label('Overall F1 Score', fontsize=12)

# Add labels and title
plt.xlabel('Number of Frames', fontsize=14)
plt.ylabel('Normalized Standard Deviation (NSD)', fontsize=14)
plt.title('Number of Frames vs NSD with Overall F1 Score Color', fontsize=16)

# Add grid
plt.grid(True, linestyle='--', alpha=0.6)

# Add annotations for key points
top_f1 = df['overall_f1'].idxmax()
plt.annotate('Best F1',
             xy=(df.loc[top_f1, 'total_frames'], df.loc[top_f1, 'nsd']),
             xytext=(10, 10),
             textcoords='offset points',
             arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=.2'))

# Improve layout
plt.tight_layout()

# Save the figure
plt.savefig('frames_nsd_f1_plot.png', dpi=300)

# Show the plot
plt.show()

In [ ]:
# Create the figure and axis
plt.figure(figsize=(12, 8))

# Create a scatter plot
scatter = plt.scatter(df['total_frames'], df['entropy'],
                      c=df['overall_f1'],
                      cmap='cividis',  # Using cividis colormap (colorblind friendly)
                      s=100,           # Size of markers
                      alpha=0.8,       # Transparency
                      edgecolors='black')

# Add a color bar
cbar = plt.colorbar(scatter)
cbar.set_label('Overall F1 Score', fontsize=12)

# Add labels and title
plt.xlabel('Number of Frames', fontsize=14)
plt.ylabel('Entropy', fontsize=14)
plt.title('Number of Frames vs Entropy with Overall F1 Score Color', fontsize=16)

# Add grid
plt.grid(True, linestyle='--', alpha=0.6)

# Add a horizontal line at maximum entropy (2.0)
plt.axhline(y=2.0, color='red', linestyle='--', alpha=0.5)
plt.text(plt.xlim()[0], 2.02, 'Max Entropy (2.0)', va='bottom', ha='left',
         bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="red", alpha=0.7))

# Improve layout
plt.tight_layout()

# Save the figure
plt.savefig('frames_entropy_f1_plot.png', dpi=300)

# Show the plot
plt.show()